# 🏦 Nexopus Finance Ops — Fine-tuning Contábil

**Modelo base:** `Qwen/Qwen2.5-7B-Instruct`  
**Técnica:** QLoRA via Unsloth (2x mais rápido, 70% menos VRAM)  
**GPU recomendada:** T4 16GB (grátis no Colab) ou A100 (RunPod)  
**Saída:** modelo GGUF quantizado para rodar no seu servidor via Ollama  

### Antes de começar
1. Menu → **Runtime → Change runtime type → T4 GPU**
2. Execute as células em ordem
3. O modelo final (~4 GB) será salvo no Google Drive

## 1. Instalar dependências

In [ ]:
# Verifica GPU disponível
!nvidia-smi
!nvcc --version

In [ ]:
%%capture
!pip install unsloth==2025.4.7
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install datasets transformers trl peft bitsandbytes accelerate
!pip install huggingface_hub

## 2. Carregar modelo base com Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto-detecta (float16 no T4, bfloat16 no A100)
load_in_4bit = True  # QLoRA: menos VRAM, mais velocidade

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("✅ Modelo carregado com sucesso")

## 3. Configurar adaptador LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # Rank LoRA — 16 é bom equilíbrio qualidade/tamanho
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,          # Optimizado pelo Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
)

# Resumo do modelo
model.print_trainable_parameters()

## 4. Dataset Contábil (Nexopus Finance)

In [ ]:
# Dataset de classificação de contas contábeis no padrão brasileiro
# Baseado em NBC TG / CFC

ACCOUNTING_DATASET = [
    # Receitas
    {"conta": "Receita de Vendas de Mercadorias", "tipo": "receita", "natureza": "credora", "grupo": "receita_operacional"},
    {"conta": "Receita de Prestação de Serviços", "tipo": "receita", "natureza": "credora", "grupo": "receita_operacional"},
    {"conta": "Receita Financeira", "tipo": "receita", "natureza": "credora", "grupo": "receita_financeira"},
    {"conta": "Receita com Aluguéis", "tipo": "receita", "natureza": "credora", "grupo": "outras_receitas"},
    {"conta": "Receita de Dividendos", "tipo": "receita", "natureza": "credora", "grupo": "receita_financeira"},
    # Deduções da Receita
    {"conta": "Devoluções de Vendas", "tipo": "deducao_receita", "natureza": "devedora", "grupo": "deducoes_receita"},
    {"conta": "ICMS sobre Vendas", "tipo": "deducao_receita", "natureza": "devedora", "grupo": "deducoes_receita"},
    {"conta": "PIS sobre Faturamento", "tipo": "deducao_receita", "natureza": "devedora", "grupo": "deducoes_receita"},
    {"conta": "COFINS sobre Faturamento", "tipo": "deducao_receita", "natureza": "devedora", "grupo": "deducoes_receita"},
    {"conta": "ISS sobre Serviços", "tipo": "deducao_receita", "natureza": "devedora", "grupo": "deducoes_receita"},
    # CMV / CPV
    {"conta": "Custo das Mercadorias Vendidas", "tipo": "custo", "natureza": "devedora", "grupo": "cmv"},
    {"conta": "Custo dos Produtos Vendidos", "tipo": "custo", "natureza": "devedora", "grupo": "cmv"},
    {"conta": "Custo dos Serviços Prestados", "tipo": "custo", "natureza": "devedora", "grupo": "cmv"},
    # Despesas Operacionais
    {"conta": "Despesas com Salários", "tipo": "despesa", "natureza": "devedora", "grupo": "despesa_pessoal"},
    {"conta": "Despesas com Encargos Sociais", "tipo": "despesa", "natureza": "devedora", "grupo": "despesa_pessoal"},
    {"conta": "Despesas com Aluguéis", "tipo": "despesa", "natureza": "devedora", "grupo": "despesa_administrativa"},
    {"conta": "Despesas com Energia Elétrica", "tipo": "despesa", "natureza": "devedora", "grupo": "despesa_administrativa"},
    {"conta": "Despesas com Marketing", "tipo": "despesa", "natureza": "devedora", "grupo": "despesa_comercial"},
    {"conta": "Despesas com Depreciação", "tipo": "despesa", "natureza": "devedora", "grupo": "despesa_administrativa"},
    {"conta": "Despesas Financeiras", "tipo": "despesa", "natureza": "devedora", "grupo": "despesa_financeira"},
    {"conta": "Juros sobre Empréstimos", "tipo": "despesa", "natureza": "devedora", "grupo": "despesa_financeira"},
    # Ativos
    {"conta": "Caixa e Equivalentes de Caixa", "tipo": "ativo", "natureza": "devedora", "grupo": "ativo_circulante"},
    {"conta": "Contas a Receber de Clientes", "tipo": "ativo", "natureza": "devedora", "grupo": "ativo_circulante"},
    {"conta": "Estoque de Mercadorias", "tipo": "ativo", "natureza": "devedora", "grupo": "ativo_circulante"},
    {"conta": "Adiantamento a Fornecedores", "tipo": "ativo", "natureza": "devedora", "grupo": "ativo_circulante"},
    {"conta": "Imóveis", "tipo": "ativo", "natureza": "devedora", "grupo": "ativo_nao_circulante"},
    {"conta": "Máquinas e Equipamentos", "tipo": "ativo", "natureza": "devedora", "grupo": "ativo_nao_circulante"},
    {"conta": "Veículos", "tipo": "ativo", "natureza": "devedora", "grupo": "ativo_nao_circulante"},
    {"conta": "Participações em Coligadas", "tipo": "ativo", "natureza": "devedora", "grupo": "ativo_nao_circulante"},
    # Passivos
    {"conta": "Fornecedores a Pagar", "tipo": "passivo", "natureza": "credora", "grupo": "passivo_circulante"},
    {"conta": "Salários a Pagar", "tipo": "passivo", "natureza": "credora", "grupo": "passivo_circulante"},
    {"conta": "Impostos a Recolher", "tipo": "passivo", "natureza": "credora", "grupo": "passivo_circulante"},
    {"conta": "INSS a Recolher", "tipo": "passivo", "natureza": "credora", "grupo": "passivo_circulante"},
    {"conta": "FGTS a Recolher", "tipo": "passivo", "natureza": "credora", "grupo": "passivo_circulante"},
    {"conta": "Empréstimos Bancários CP", "tipo": "passivo", "natureza": "credora", "grupo": "passivo_circulante"},
    {"conta": "Financiamentos LP", "tipo": "passivo", "natureza": "credora", "grupo": "passivo_nao_circulante"},
    {"conta": "Debêntures", "tipo": "passivo", "natureza": "credora", "grupo": "passivo_nao_circulante"},
    # Patrimônio Líquido
    {"conta": "Capital Social", "tipo": "pl", "natureza": "credora", "grupo": "patrimonio_liquido"},
    {"conta": "Reserva Legal", "tipo": "pl", "natureza": "credora", "grupo": "patrimonio_liquido"},
    {"conta": "Reserva de Lucros", "tipo": "pl", "natureza": "credora", "grupo": "patrimonio_liquido"},
    {"conta": "Lucros Acumulados", "tipo": "pl", "natureza": "credora", "grupo": "patrimonio_liquido"},
    {"conta": "Prejuízos Acumulados", "tipo": "pl", "natureza": "devedora", "grupo": "patrimonio_liquido"},
]

print(f"✅ {len(ACCOUNTING_DATASET)} exemplos no dataset")

In [ ]:
import json

# Template de prompt para classificação contábil
SYSTEM_PROMPT = """Você é um especialista em contabilidade brasileira, seguindo as normas NBC TG e CFC.
Dado o nome de uma conta contábil, classifique-a em JSON com os campos:
- tipo: ativo | passivo | receita | despesa | deducao_receita | custo | pl
- natureza: devedora | credora
- grupo: grupo específico da conta
Responda APENAS com JSON válido, sem explicações adicionais."""

def format_example(item):
    user_msg = f"Classifique a conta contábil: {item['conta']}"
    assistant_msg = json.dumps({
        "tipo": item["tipo"],
        "natureza": item["natureza"],
        "grupo": item["grupo"]
    }, ensure_ascii=False)
    return {
        "text": tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": assistant_msg},
            ],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

formatted_data = [format_example(item) for item in ACCOUNTING_DATASET]

from datasets import Dataset
dataset = Dataset.from_list(formatted_data)
print(f"✅ Dataset formatado: {len(dataset)} exemplos")
print("\nExemplo de entrada:")
print(dataset[0]["text"][:500])

## 5. Treinar o modelo

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,          # 3 épocas para dataset pequeno
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="/tmp/nexopus-finetune",
    ),
)

# Inicia o treinamento
trainer_stats = trainer.train()
print(f"✅ Treinamento concluído! Loss final: {trainer_stats.training_loss:.4f}")

## 6. Testar o modelo treinado

In [ ]:
FastLanguageModel.for_inference(model)  # Habilita modo inferência otimizado

def classify_account(account_name: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Classifique a conta contábil: {account_name}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=128,
        temperature=0.1,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    try:
        return json.loads(response.strip())
    except json.JSONDecodeError:
        return {"raw": response.strip()}

# Testes
test_accounts = [
    "Receita de Vendas Online",
    "Despesa com Software SaaS",
    "Empréstimo Bancário de Longo Prazo",
    "Máquinas de Produção",
    "Capital Social Integralizado",
]

print("=== Resultados da Classificação ===")
for acc in test_accounts:
    result = classify_account(acc)
    print(f"\n📊 {acc}")
    print(f"   → {json.dumps(result, ensure_ascii=False)}")

## 7. Exportar para GGUF (rodar no seu servidor via Ollama)

In [ ]:
# Salva adaptador LoRA (pequeno, só os pesos extras)
model.save_pretrained("/tmp/nexopus-lora-adapter")
tokenizer.save_pretrained("/tmp/nexopus-lora-adapter")
print("✅ Adaptador LoRA salvo")

# Exporta para GGUF Q4_K_M (melhor equilíbrio qualidade/tamanho, ~4.5 GB)
model.save_pretrained_gguf(
    "/tmp/nexopus-gguf",
    tokenizer,
    quantization_method="q4_k_m",  # Recomendado para CPU
)
print("✅ Arquivo GGUF exportado em /tmp/nexopus-gguf/")
!ls -lh /tmp/nexopus-gguf/

In [ ]:
# Salva no Google Drive para não perder ao fechar o Colab
from google.colab import drive
drive.mount("/content/drive")

import shutil, os
dest = "/content/drive/MyDrive/nexopus-finetune"
os.makedirs(dest, exist_ok=True)

# Copia GGUF para o Drive
shutil.copytree("/tmp/nexopus-gguf", f"{dest}/gguf", dirs_exist_ok=True)
shutil.copytree("/tmp/nexopus-lora-adapter", f"{dest}/lora-adapter", dirs_exist_ok=True)

print(f"✅ Modelo salvo em: {dest}")
!ls -lh "{dest}/gguf/"

## 8. Usar o modelo no seu servidor (após download)

Após baixar o arquivo `.gguf` do Google Drive para o seu servidor:

```bash
# Cria um Modelfile para o Ollama
cat > /tmp/Modelfile <<'EOF'
FROM /caminho/para/nexopus-qwen2.5-7b-q4_k_m.gguf

SYSTEM """Você é um especialista em contabilidade brasileira, seguindo as normas NBC TG e CFC.
Dado o nome de uma conta contábil, classifique-a em JSON com os campos:
- tipo: ativo | passivo | receita | despesa | deducao_receita | custo | pl
- natureza: devedora | credora
- grupo: grupo específico da conta
Responda APENAS com JSON válido, sem explicações adicionais."""

PARAMETER temperature 0.1
PARAMETER num_ctx 2048
EOF

# Registra o modelo no Ollama
ollama create nexopus-accounting -f /tmp/Modelfile

# Testa
ollama run nexopus-accounting "Classifique a conta contábil: Receita de Vendas"
```

No projeto, altere `.env`:
```env
LLM_PROVIDER=ollama
OLLAMA_BASE_URL=http://localhost:11434/v1
LLM_MODEL=nexopus-accounting
```